# 02 - Silver Transformations

Creates a clean trusted Silver Delta dataset from the Bronze layer.

In [ ]:
%run ../utilities/config

In [ ]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

silver_df = (
    bronze_df
    .dropDuplicates(["event_id"])
    .filter(col("event_id").isNotNull())
    .filter(col("user_id").isNotNull())
    .filter(col("event_type").isin(VALID_EVENT_TYPES))
    .withColumn("event_timestamp", to_timestamp(col("event_time")))
    .withColumn("silver_processed_timestamp", current_timestamp())
)

silver_df.write.format("delta").mode("overwrite").save(SILVER_PATH)

In [ ]:
display(spark.read.format("delta").load(SILVER_PATH))

print("Bronze count:", spark.read.format("delta").load(BRONZE_PATH).count())
print("Silver count:", spark.read.format("delta").load(SILVER_PATH).count())